In [ ]:
!pip install ultralytics fastapi uvicorn python-multipart opencv-python-headless pandas matplotlib requests -q

In [ ]:
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import json
import uuid

from ultralytics import YOLO
from PIL import Image
from google.colab import files
from IPython.display import display

print("Libraries imported successfully")
print("CUDA Available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Libraries imported successfully
CUDA Available: False


In [ ]:
model = YOLO("yolov8n.pt")

print("Model loaded successfully")
print("Model: YOLOv8 Nano")
print("Task:", model.task)

Model loaded successfully
Model: YOLOv8 Nano
Task: detect


In [ ]:
uploaded = files.upload()

image_path = next(iter(uploaded.keys()))

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.title("API Input Image")
plt.axis("off")
plt.show()

print("Request ID:", str(uuid.uuid4()))
print("Image:", image_path)
print("Image Dimensions:", image.shape)
print("Input received successfully")

In [ ]:
start_preprocessing = time.time()

original_height, original_width = image.shape[:2]

processed_image = cv2.resize(image, (640, 640))

normalized_image = processed_image.astype(np.float32) / 255.0

preprocessing_time = time.time() - start_preprocessing

plt.figure(figsize=(8, 5))
plt.imshow(normalized_image)
plt.title("Preprocessed Image (640 × 640)")
plt.axis("off")
plt.show()

print("Original Dimensions:", original_width, "x", original_height)
print("Processed Dimensions:", processed_image.shape)
print("Preprocessing Time:", round(preprocessing_time * 1000, 2), "ms")

In [ ]:
start_inference = time.time()

results = model.predict(
    source=image,
    conf=0.25,
    imgsz=640,
    verbose=False
)

inference_time = time.time() - start_inference

print("Model execution completed")
print("Inference Time:", round(inference_time * 1000, 2), "ms")

In [ ]:
detections = []

for result in results:
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append({
            "Object": model.names[class_id],
            "Confidence": round(confidence, 4),
            "X1": round(x1, 2),
            "Y1": round(y1, 2),
            "X2": round(x2, 2),
            "Y2": round(y2, 2)
        })

detection_df = pd.DataFrame(detections)

display(detection_df)

print("Total Objects Detected:", len(detections))

In [ ]:
annotated_image = results[0].plot()

annotated_image = cv2.cvtColor(
    annotated_image,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(10, 6))
plt.imshow(annotated_image)
plt.title("Model Detection Output")
plt.axis("off")
plt.show()

In [ ]:
response = {
    "status": "success",
    "request_id": str(uuid.uuid4()),
    "model": "YOLOv8n",
    "inference_time_ms": round(inference_time * 1000, 2),
    "preprocessing_time_ms": round(preprocessing_time * 1000, 2),
    "total_detections": len(detections),
    "detections": detections
}

print(json.dumps(response, indent=4))

In [ ]:
stages = [
    "API Input",
    "Preprocessing",
    "Model Execution",
    "Response Generation"
]

times = [
    0,
    preprocessing_time * 1000,
    inference_time * 1000,
    0
]

plt.figure(figsize=(10, 5))
plt.bar(stages, times)

plt.title("Inference Request Lifecycle")
plt.ylabel("Execution Time (ms)")
plt.xlabel("Processing Stages")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.show()

print("INFERENCE LIFECYCLE COMPLETED")
print("Input → Preprocessing → Model → Response")